# Korean Listed Company Valuation Automation
## Comparable Company Analysis (Comps)

### Goal Index
1. Library Imports & Setup
2. Data Collection (DART API)
3. D&A Extraction (HTML Parsing)
4. Valuation Metrics Calculation
5. Excel Output

In [1]:
import pandas as pd

### 커널 재등록하면서 캐시 오류 날 경우 아래 코드들 실행 
### (커널은 Anaconda Python 기준이어야 함)

In [11]:
# If Kernel cache error occurs, run the cells below
# (Kernel must be set to Anaconda Python)

import os
import glob

# 다양한 경로 탐색
paths = [
    os.path.expanduser('~') + '/**/*opendart*',
    'C:/Users/cogus/**/*opendart*',
]

for p in paths:
    files = glob.glob(p, recursive=True)
    for f in files:
        print(f)

C:\Users\cogus\anaconda3\Lib\site-packages\OpenDartReader
C:\Users\cogus\anaconda3\Lib\site-packages\OpenDartReader-0.2.3.dist-info
C:\Users\cogus\AppData\Local\Programs\Python\Python312\Lib\site-packages\OpenDartReader
C:\Users\cogus\AppData\Local\Programs\Python\Python312\Lib\site-packages\OpenDartReader-0.2.3.dist-info
C:\Users\cogus\docs_cache\opendartreader_corp_codes_20260405.pkl
C:\Users\cogus\OneDrive\바탕 화면\Kearney Data\docs_cache\opendartreader_corp_codes_20260327.pkl
C:/Users/cogus\anaconda3\Lib\site-packages\OpenDartReader
C:/Users/cogus\anaconda3\Lib\site-packages\OpenDartReader-0.2.3.dist-info
C:/Users/cogus\AppData\Local\Programs\Python\Python312\Lib\site-packages\OpenDartReader
C:/Users/cogus\AppData\Local\Programs\Python\Python312\Lib\site-packages\OpenDartReader-0.2.3.dist-info
C:/Users/cogus\docs_cache\opendartreader_corp_codes_20260405.pkl
C:/Users/cogus\OneDrive\바탕 화면\Kearney Data\docs_cache\opendartreader_corp_codes_20260327.pkl


In [13]:
import os

files = [
    r'C:\Users\cogus\docs_cache\opendartreader_corp_codes_20260405.pkl',
    r'C:\Users\cogus\OneDrive\바탕 화면\Kearney Data\docs_cache\opendartreader_corp_codes_20260327.pkl'
]

for f in files:
    if os.path.exists(f):
        os.remove(f)
        print(f"삭제: {f}")

print("완료")

삭제: C:\Users\cogus\docs_cache\opendartreader_corp_codes_20260405.pkl
삭제: C:\Users\cogus\OneDrive\바탕 화면\Kearney Data\docs_cache\opendartreader_corp_codes_20260327.pkl
완료


## [Development Notes]
### Data Engineering Challenges & Solutions

Samsung Electronics (005930) selected as the base case for code development - FY2023 annual report data
This pipeline supports any fiscal year available on DART

In [42]:
## 삼성전자 재무제표 불러오기
api_key = 'dfd644cb97b2f1b8d21fa57b07a0d90a0760dd07' #개인 발급
dart = OpenDartReader(api_key)

# 삼성전자(005930) 2023 재무제표 - 요약 재무제표 API (.finstate())
df = dart.finstate('005930', 2023) #여기 rcept_no에 사업보고서 접수번호 존재

# df 구조 확인
print(df.columns.tolist())
print(df.head())

['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'stock_code', 'fs_div', 'fs_nm', 'sj_div', 'sj_nm', 'account_nm', 'thstrm_nm', 'thstrm_dt', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_dt', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_dt', 'bfefrmtrm_amount', 'ord', 'currency']
         rcept_no reprt_code bsns_year corp_code stock_code fs_div   fs_nm  \
0  20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
1  20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
2  20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
3  20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   
4  20240312000736      11011      2023  00126380     005930    CFS  연결재무제표   

  sj_div  sj_nm account_nm  ...      thstrm_dt        thstrm_amount frmtrm_nm  \
0     BS  재무상태표       유동자산  ...  2023.12.31 현재  195,936,557,000,000    제 54 기   
1     BS  재무상태표      비유동자산  ...  2023.12.31 현재  259,969,423,000,000    제 54 기   
2     BS  재무

In [44]:
# 연결재무제표만 필터 (삼성전자+자회사 전부 합친 것) #df 컬럼 보고 감 잡을 수 있음
cfs = df[df['fs_div'] == 'CFS']

# 계정 목록 확인. 순서대로 재무제표종류, 계정 이름, 당기(2023년) 금액. 계정이 main.
print(cfs[['sj_nm', 'account_nm', 'thstrm_amount']].to_string())

    sj_nm  account_nm        thstrm_amount
0   재무상태표        유동자산  195,936,557,000,000
1   재무상태표       비유동자산  259,969,423,000,000
2   재무상태표        자산총계  455,905,980,000,000
3   재무상태표        유동부채   75,719,452,000,000
4   재무상태표       비유동부채   16,508,663,000,000
5   재무상태표        부채총계   92,228,115,000,000
6   재무상태표         자본금      897,514,000,000
7   재무상태표       이익잉여금  346,652,238,000,000
8   재무상태표        자본총계  363,677,865,000,000
9   손익계산서         매출액  258,935,494,000,000
10  손익계산서        영업이익    6,566,976,000,000
11  손익계산서  법인세차감전 순이익   11,006,265,000,000
12  손익계산서   당기순이익(손실)   15,487,100,000,000
13  손익계산서   당기순이익(손실)   15,487,100,000,000
14  손익계산서       총포괄손익   18,837,411,000,000


In [17]:
# 출력 결과를 봤을 때 매출액, 영업이익(EBIT), 당기순이익, 자산/부채/자본 존재하나 감가상각비가 없음
# 다른 API를 통해 현금흐름표에서 따로 뽑기로 함. 일단 있는 것부터 정리

# 손익계산서 - 1년 동안 얼마 벌고 썼는지 (매출액, 영업이익, 당기순이익 등)
# 재무상태표 - 지금 이 순간 뭘 갖고 있고 뭘 빚졌는지 (자산, 부채, 자본 등)

Revenue, EBIT, net income, and balance sheet items confirmed. D&A not available via summary API
(to be extracted separately from the annual report filing)

In [19]:
# 필요한 계정만 뽑기
income = cfs[cfs['sj_nm'] == '손익계산서']
balance = cfs[cfs['sj_nm'] == '재무상태표']

# 매출액, 영업이익 추출
revenue = int(income[income['account_nm'] == '매출액']['thstrm_amount'].values[0].replace(',', ''))
ebit = int(income[income['account_nm'] == '영업이익']['thstrm_amount'].values[0].replace(',', ''))
net_income = int(income[income['account_nm'].str.contains('당기순이익')]['thstrm_amount'].values[0].replace(',', ''))

print(f"매출액: {revenue/1e12:.1f}조")
print(f"영업이익(EBIT): {ebit/1e12:.1f}조")
print(f"당기순이익: {net_income/1e12:.1f}조")

매출액: 258.9조
영업이익(EBIT): 6.6조
당기순이익: 15.5조


Deriving EV

In [21]:
# 재무상태표 계정 전체 확인: 현금이랑 차입금 뽑아야 EV 계산 가능 
print(balance[['account_nm', 'thstrm_amount']].to_string())

  account_nm        thstrm_amount
0       유동자산  195,936,557,000,000
1      비유동자산  259,969,423,000,000
2       자산총계  455,905,980,000,000
3       유동부채   75,719,452,000,000
4      비유동부채   16,508,663,000,000
5       부채총계   92,228,115,000,000
6        자본금      897,514,000,000
7      이익잉여금  346,652,238,000,000
8       자본총계  363,677,865,000,000


In [23]:
# 세부 재무제표 불러오기 (위에 걸로는 파악 불가)
df_detail = dart.finstate_all('005930', 2023, reprt_code='11011') #DART 개발가이드에서 정해놓은 코드

# 재무상태표만 필터 (sj_div == 'BS')
bs_detail = df_detail[df_detail['sj_div'] == 'BS']

# 계정 전체 확인
print(bs_detail[['account_nm', 'thstrm_amount']].to_string())

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
                account_nm    thstrm_amount
0                     자산총계  455905980000000
1                     유동자산  195936557000000
2                      미수금    6633248000000
3                     선급비용    3366130000000
4                 현금및현금성자산   69080893000000
5              단기상각후원가금융자산     608281000000
6          단기당기손익-공정가치금융자산      27112000000
7                     매출채권   36647393000000
8                     재고자산   51625874000000
9                 매각예정분류자산     217864000000
10                  기타유동자산    5038838000000
11                  단기금융상품   22690924000000
12                   비유동자산  259969423000000
13                 이연법인세자산   10211797000000
14                    무형자산   22741862000000
15          관계종속기업투자자산-지분법   11767444000000
16           당기손익-공정가치금융자산    1431394000000
17  기타포괄손익-공정가치 측정 비유동금융자산    7481297000000
18                 순확정급여자산    4905219000000
19                 기타비유동자산   14174148000000
20                    유형자산

In [25]:
# 숫자 변환 함수
def get_amount(df, account_name):
    row = df[df['account_nm'] == account_name]
    if len(row) == 0:
        return 0
    return int(row['thstrm_amount'].values[0])

# 재무상태표 항목 추출
cash = get_amount(bs_detail, '현금및현금성자산')
short_fin = get_amount(bs_detail, '단기금융상품')
short_debt = get_amount(bs_detail, '단기차입금')
current_ltd = get_amount(bs_detail, '유동성장기부채')
bonds = get_amount(bs_detail, '사채')
long_debt = get_amount(bs_detail, '장기차입금')

# 순부채 계산
total_debt = short_debt + current_ltd + bonds + long_debt
net_debt = total_debt - cash - short_fin

print(f"현금: {cash/1e12:.1f}조")
print(f"단기금융상품: {short_fin/1e12:.1f}조")
print(f"총차입금: {total_debt/1e12:.1f}조")
print(f"순부채: {net_debt/1e12:.1f}조")

현금: 69.1조
단기금융상품: 22.7조
총차입금: 12.7조
순부채: -79.1조


In [29]:
# 순부채 계산 완료. 이제 시가총액 필요

import FinanceDataReader as fdr

# 삼성전자 주가
stock = fdr.DataReader('005930', '2023-12-28', '2023-12-28') #29일 했을 때 비어 있었음
print(stock)

             Open   High    Low  Close    Volume   Change
Date                                                     
2023-12-28  77700  78500  77500  78500  17797536  0.00641


In [35]:
# 발행주식수 가져오기

# DART 주식총수 현황으로 가져오기
shares = dart.report('005930', '주식총수', 2023)
print(shares)

         rcept_no corp_cls corp_code corp_name   se isu_stock_totqy  \
0  20240312000736        Y  00126380      삼성전자  보통주  20,000,000,000   
1  20240312000736        Y  00126380      삼성전자  우선주   5,000,000,000   
2  20240312000736        Y  00126380      삼성전자   합계  25,000,000,000   
3  20240312000736        Y  00126380      삼성전자   비고               -   

  now_to_isu_stock_totqy now_to_dcrs_stock_totqy redc   profit_incnr  \
0          7,780,466,850           1,810,684,300    -  1,810,684,300   
1          1,194,671,350             371,784,650    -    371,784,650   
2          8,975,138,200           2,182,468,950    -  2,182,468,950   
3                      -                       -    -          자사주소각   

  rdmstk_repy etc     istc_totqy tesstk_co distb_stock_co     stlm_dt  
0           -   -  5,969,782,550         -  5,969,782,550  2023-12-31  
1           -   -    822,886,700         -    822,886,700  2023-12-31  
2           -   -  6,792,669,250         -  6,792,669,250  2023-12-

In [37]:
# 발행주식수 자동 추출
shares_common = int(shares[shares['se'] == '보통주']['istc_totqy'].values[0].replace(',', ''))
shares_preferred = int(shares[shares['se'] == '우선주']['istc_totqy'].values[0].replace(',', ''))
total_shares = shares_common + shares_preferred

# 시가총액 재계산
market_cap = close_price * total_shares

print(f"보통주: {shares_common:,}주")
print(f"우선주: {shares_preferred:,}주")
print(f"시가총액: {market_cap/1e12:.1f}조")

보통주: 5,969,782,550주
우선주: 822,886,700주
시가총액: 533.2조


In [31]:
# 주가
close_price = stock['Close'].values[0]

# 발행주식수 (삼성전자 보통주 + 우선주)
shares_common = 5969782550  # 보통주
shares_preferred = 822886700  # 우선주
total_shares = shares_common + shares_preferred

# 시가총액
market_cap = close_price * total_shares

print(f"주가: {close_price:,}원")
print(f"시가총액: {market_cap/1e12:.1f}조")

주가: 78,500원
시가총액: 533.2조


In [39]:
## 이제 EV 계산!
# 순부채 (아까 계산한 값)
net_debt = total_debt - cash - short_fin

# EV 계산
EV = market_cap + net_debt

print(f"시가총액: {market_cap/1e12:.1f}조")
print(f"순부채: {net_debt/1e12:.1f}조")
print(f"EV: {EV/1e12:.1f}조")

시가총액: 533.2조
순부채: -79.1조
EV: 454.1조


### EV (market cap + net debt), EBIT, revenue, and net income extracted. 
### Moving on to D&A extraction for EBITDA calculation.

In [41]:
## EBITDA 계산 위해 감가상각비 가져옴

# 현금흐름표 확인
cf_detail = df_detail[df_detail['sj_div'] == 'CF']
print(cf_detail[['account_nm', 'thstrm_amount']].to_string())

                        account_nm    thstrm_amount
83                      기초현금및현금성자산   49680710000000
84                      기말현금및현금성자산   69080893000000
85                          매각예정분류     -14153000000
86                        재무활동현금흐름   -8593059000000
87                 장기차입금의 차입 (주27)     354712000000
88            사채 및 장기차입금의 상환 (주27)    1219579000000
89                       비지배지분의 증감      -9118000000
90            단기차입금의 순증가(감소) (주27)    2145400000000
91                         배당금의 지급    9864474000000
92                        투자활동현금흐름  -16922817000000
93               당기손익-공정가치금융자산의 처분      63962000000
94            기타포괄손익-공정가치측정금융자산의처분    6521568000000
95                      장기금융상품의 처분    4565426000000
96   매각예정으로 분류된 비유동자산이나 처분자산집단의 처분                0
97               당기손익-공정가치금융자산의 취득     130459000000
98            기타포괄손익-공정가치측정금융자산의취득     124488000000
99                      장기금융상품의 취득    5307770000000
100           단기상각후원가금융자산의 순감소(증가)    -195616000000
101       단기

In [43]:
# '감가' 또는 '상각' 포함된 계정 찾기
da = df_detail[df_detail['account_nm'].str.contains('감가|상각|depreciation', case=False, na=False)]
print(da[['account_nm', 'thstrm_amount']].to_string())

               account_nm  thstrm_amount
5             단기상각후원가금융자산   608281000000
100  단기상각후원가금융자산의 순감소(증가)  -195616000000


감가상각비가 DART 요약 데이터에 없음. 사업보고서 원문에 주석으로 들어가 있어서 API로 못 가져옴.
1) EBIT으로 대신해야 하나?
2) 아니면 사업보고서 원문 주석으로 파악해야 (HTML 파싱)

### DART 사업보고서 원문 파싱을 해봅시다. (블룸버그 터미널 쓰면 사실 한 번에 나옴, 우린 못 쓰니까) / HTML 트리 안에 있는 테이블에 D&A 숫자가 숨어 있음. (커니에서 크롤링 업무할 때 CSS SELECTOR 하던 거랑 로직이 똑같음!)

HTML 전체를 읽어서 테이블 다 뒤지고 감가상각비 숫자만 뽑아내는 거. 그게 파싱. (구조화되지 않은 테이블에서 필요한 정보를 추출하는 거)

### Parsing DART annual report HTML to extract D&A
### Same logic as CSS selector-based crawling — reading the HTML tree and extracting target values from nested tables

## Challenge: D&A Not Available via API

- D&A is absent from DART's summary financials. It is embedded in the footnotes of the annual report filing, not exposed through the standard API.
- Options considered: (1) fall back to EBIT-only multiples, or (2) parse the raw HTML filing directly
### - Decided to parse the annual report HTML, replicating what Bloomberg Terminal provides out of the box.

In [51]:
# 삼성전자 2023 사업보고서 접수번호
rcept_no = '20240312000736'  # 아까 데이터에서 나왔던 거

# 보고서 문서 목록
docs = dart.sub_docs(rcept_no)
print(docs[['title', 'url']].to_string())

                          title                                                                                                                             url
0                     사 업 보 고 서          http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=1&offset=957&length=4063&dtd=dart4.xsd
1                【 대표이사 등의 확인 】         http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=2&offset=28204&length=428&dtd=dart4.xsd
2                     I. 회사의 개요      http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=3&offset=28636&length=118735&dtd=dart4.xsd
3                     1. 회사의 개요       http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=4&offset=28746&length=24303&dtd=dart4.xsd
4                     2. 회사의 연혁       http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=5&offset=53053&length=20277&dtd=dart4.xsd
5                   3. 자본금 변동사항        h

In [53]:
# 19번 파싱
import requests
from bs4 import BeautifulSoup

# 연결재무제표 주석 URL
url = 'http://dart.fss.or.kr/report/viewer.do?rcpNo=20240312000736&dcmNo=9702846&eleId=25&offset=430817&length=1503831&dtd=dart4.xsd'

response = requests.get(url)
soup = BeautifulSoup(response.content, 'lxml')

# 감가상각 관련 텍스트 찾기
tables = soup.find_all('table')
print(f"테이블 수: {len(tables)}")

테이블 수: 683


In [59]:
# 감가상각비 포함 테이블 번호만 (감가상각으로 서치해서 다 뽑아냈을 땐 너무 많았음)
found = []
for i, table in enumerate(tables):
    if '감가상각비' in table.get_text():
        found.append(i)

print(f"총 {len(found)}개 테이블")
print(found)

총 13개 테이블
[217, 220, 247, 250, 429, 432, 438, 441, 517, 520, 630, 633, 636]


In [61]:
# 테이블 하나씩 확인해봅시다 이제
# 217, 220번 테이블 확인
for i in [217, 220]:
    print(f"=== 테이블 {i} ===")
    print(tables[i].get_text()[:800])
    print()

=== 테이블 217 ===












　

토지


건물 및 구축물


기계장치


건설중인자산


기타


유형자산  합계






기초 유형자산


9,892,167


40,706,918


79,714,631


33,607,564


4,124,108


168,045,388




일반취득 및 자본적지출


172,262


6,498,611


33,641,691


13,141,766


1,462,032


54,916,362




사업결합을 통한 취득, 유형자산


0


18,125


20,140


34,698


165


73,128




감가상각비, 유형자산


(49,367)


(3,884,333)


(30,031,617)


0


(1,567,094)


(35,532,411)




처분ㆍ폐기


(25,934)


(181,700)


(37,681)


(256)


(30,547)


(276,118)




손상(환입)


0


(30,864)


(47,044)


0


(7,449)


(85,357)




매각예정분류


(6,615)


(54,318)


(37,101)


(6,255)


(14,100)


(118,389)




기타


16,864


165,676


86,149


(57,189)


22,159


233,659




기말 유형자산


9,999,377


43,238,115


83,309,168


46,720,328


3,989,274


187,256,262





=== 테이블 220 ===












　

토지


건물 및 구축물


기계장치


건설중인자산


기타


유형자산  합계






기초 유형자산


9,830,154


38,869,440


79,526,297


18,009,324


3,693,324


149,928,539




일반취득 및 자본적지출


138,925


5,302,095


31,0

In [63]:
# 오 찾음. 근데 사업부별로 나뉜듯...? 자세히 보기

# 217번 테이블 전체 확인
df_table = pd.read_html(str(tables[217]))[0]
print(df_table.to_string())

          Unnamed: 0        토지     건물 및 구축물          기계장치    건설중인자산           기타       유형자산 합계
0            기초 유형자산   9892167     40706918      79714631  33607564      4124108     168045388
1       일반취득 및 자본적지출    172262      6498611      33641691  13141766      1462032      54916362
2  사업결합을 통한 취득, 유형자산         0        18125         20140     34698          165         73128
3        감가상각비, 유형자산  (49,367)  (3,884,333)  (30,031,617)         0  (1,567,094)  (35,532,411)
4              처분ㆍ폐기  (25,934)    (181,700)      (37,681)     (256)     (30,547)     (276,118)
5             손상(환입)         0     (30,864)      (47,044)         0      (7,449)      (85,357)
6             매각예정분류   (6,615)     (54,318)      (37,101)   (6,255)     (14,100)     (118,389)
7                 기타     16864       165676         86149  (57,189)        22159        233659
8            기말 유형자산   9999377     43238115      83309168  46720328      3989274     187256262


C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\3821787846.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_table = pd.read_html(str(tables[217]))[0]


In [67]:
# (감가상각비, 유형자산 / 유형자산 합계) 35532411 나왔음!
# 무형자산도 찾아야함. 220번은 전기 데이터인 것 같다...

# 다른 테이블도 봐보기
for i in [247, 250]:
    print(f"=== 테이블 {i} ===")
    df_t = pd.read_html(str(tables[i]))[0]
    print(df_t.to_string())
    print()

=== 테이블 247 ===
    Unnamed: 0      매출원가  판매비와관리비 등  기능별 항목 합계
0  감가상각비, 유형자산  31647926    3884485   35532411

=== 테이블 250 ===
    Unnamed: 0      매출원가  판매비와관리비 등  기능별 항목 합계
0  감가상각비, 유형자산  32285800    3666298   35952098



C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\2061730208.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(tables[i]))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\2061730208.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(tables[i]))[0]


In [69]:
# 여기서 217, 220에 한 걸 간단하게 구할 수 있었다. 
# 무형자산...

for i in [429, 432]:
    print(f"=== 테이블 {i} ===")
    df_t = pd.read_html(str(tables[i]))[0]
    print(df_t.to_string())
    print()

=== 테이블 429 ===
               Unnamed: 0       공시금액
0          제품 및 재공품 등의 변동  (644,905)
1   원재료 등의 사용액 및 상품 매입액 등   96219181
2                      급여   30405245
3                    퇴직급여    1157422
4                   감가상각비   35532411
5                 무형자산상각비    3134148
6                   복리후생비    6472979
7                   유틸리티비    7502408
8                   외주용역비    7058833
9                   광고선전비    5213896
10                  판매촉진비    6894395
11                  기타 비용   53422505
12              성격별 비용 합계  252368518

=== 테이블 432 ===
               Unnamed: 0          공시금액
0          제품 및 재공품 등의 변동  (10,355,548)
1   원재료 등의 사용액 및 상품 매입액 등     112591917
2                      급여      30078623
3                    퇴직급여       1440099
4                   감가상각비      35952098
5                 무형자산상각비       3155561
6                   복리후생비       6091626
7                   유틸리티비       6142317
8                   외주용역비       6597467
9                   광고선전비       6112951
10       

C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\963170540.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(tables[i]))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\963170540.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(tables[i]))[0]


In [71]:
## 찾았음. 429번 테이블에서. 
# 감가상각비 (유형자산): 35,532,411백만원
# 무형자산상각비: 3,134,148백만원
# D&A 합계: 38,666,559백만원 = 38.7조

# D&A 추출
da_tangible = 35532411 * 1_000_000  # 유형자산 감가상각비
da_intangible = 3134148 * 1_000_000  # 무형자산 상각비
da_total = da_tangible + da_intangible

# EBITDA = EBIT + D&A
ebitda = ebit + da_total

print(f"감가상각비(유형): {da_tangible/1e12:.1f}조")
print(f"상각비(무형): {da_intangible/1e12:.1f}조")
print(f"D&A 합계: {da_total/1e12:.1f}조")
print(f"EBIT: {ebit/1e12:.1f}조")
print(f"EBITDA: {ebitda/1e12:.1f}조")

감가상각비(유형): 35.5조
상각비(무형): 3.1조
D&A 합계: 38.7조
EBIT: 6.6조
EBITDA: 45.2조


## D&A successfully extracted via HTML parsing. Computing EV multiples.

In [68]:
# Core valuation multiples: EV/EBITA (primary), EV/EBIT, P/E (supplementary)
ev_ebitda = EV / ebitda
ev_ebit = EV / ebit
pe = market_cap / net_income

print(f"EV: {EV/1e12:.1f}조")
print(f"EBITDA: {ebitda/1e12:.1f}조")
print(f"당기순이익: {net_income/1e12:.1f}조")
print()
print(f"EV/EBITDA: {ev_ebitda:.1f}x")
print(f"EV/EBIT: {ev_ebit:.1f}x")
print(f"P/E: {pe:.1f}x")

NameError: name 'EV' is not defined

----------------------------------------------------------------

## Automated Pipeline: Wrapping the Full Process into Resuable Functions

The workflow above is now encapsulated into functions - input a ticker, and the entire data collection, parsing, and valuation calculation runs automatically.

## Try 1

In [83]:
def get_valuation(ticker, year=2023):
    # 1. 재무제표
    df = dart.finstate(ticker, year)
    cfs = df[df['fs_div'] == 'CFS']
    income = cfs[cfs['sj_nm'] == '손익계산서']
    
    df_detail = dart.finstate_all(ticker, year, reprt_code='11011')
    bs_detail = df_detail[df_detail['sj_div'] == 'BS']
    
    # 2. 손익계산서
    revenue = int(income[income['account_nm'] == '매출액']['thstrm_amount'].values[0].replace(',', ''))
    ebit = int(income[income['account_nm'] == '영업이익']['thstrm_amount'].values[0].replace(',', ''))
    net_income = int(income[income['account_nm'].str.contains('당기순이익')]['thstrm_amount'].values[0].replace(',', ''))
    
    # 3. 재무상태표
    def get_amount(df, name):
        row = df[df['account_nm'] == name]
        return int(row['thstrm_amount'].values[0]) if len(row) > 0 else 0
    
    cash = get_amount(bs_detail, '현금및현금성자산')
    short_fin = get_amount(bs_detail, '단기금융상품')
    short_debt = get_amount(bs_detail, '단기차입금')
    current_ltd = get_amount(bs_detail, '유동성장기부채')
    bonds = get_amount(bs_detail, '사채')
    long_debt = get_amount(bs_detail, '장기차입금')
    
    total_debt = short_debt + current_ltd + bonds + long_debt
    net_debt = total_debt - cash - short_fin
    
    # 4. D&A 파싱
    rcept_no = df['rcept_no'].values[0]
    docs = dart.sub_docs(rcept_no)
    note_url = docs[docs['title'].str.contains('연결재무제표 주석')]['url'].values[0]
    
    response = requests.get(note_url)
    soup = BeautifulSoup(response.content, 'lxml')
    tables = soup.find_all('table')
    
    da_total = 0
    for table in tables:
        if '감가상각비' in table.get_text() and '무형자산상각비' in table.get_text() and '성격별' in table.get_text():
            df_t = pd.read_html(str(table))[0]
            for _, row in df_t.iterrows():
                if '감가상각비' in str(row.iloc[0]) and '무형' not in str(row.iloc[0]):
                    da_tangible = int(str(row.iloc[-1]).replace(',', '').replace('(', '').replace(')', '')) * 1_000_000
                if '무형자산상각비' in str(row.iloc[0]):
                    da_intangible = int(str(row.iloc[-1]).replace(',', '').replace('(', '').replace(')', '')) * 1_000_000
            da_total = da_tangible + da_intangible
            break
    
    # 5. 주가/시총
    stock = fdr.DataReader(ticker, '2023-12-28', '2023-12-28')
    close_price = stock['Close'].values[0]
    shares = dart.report(ticker, '주식총수', year)
    shares_common = int(shares[shares['se'] == '보통주']['istc_totqy'].values[0].replace(',', ''))
    try:
        shares_preferred = int(shares[shares['se'] == '우선주']['istc_totqy'].values[0].replace(',', '').replace('-', '0'))
    except:
        shares_preferred = 0
    market_cap = close_price * (shares_common + shares_preferred)
    
    # 6. EV 및 멀티플
    EV = market_cap + net_debt
    ebitda = ebit + da_total
    
    return {
        'ticker': ticker,
        'revenue': revenue,
        'ebit': ebit,
        'ebitda': ebitda,
        'net_income': net_income,
        'market_cap': market_cap,
        'EV': EV,
        'EV/EBITDA': EV / ebitda,
        'EV/EBIT': EV / ebit,
        'P/E': market_cap / net_income
    }

# Test with Samsung
result = get_valuation('005930')
print(result)

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'


C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\967659349.py:42: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]


{'ticker': '005930', 'revenue': 258935494000000, 'ebit': 6566976000000, 'ebitda': 45233535000000, 'net_income': 15487100000000, 'market_cap': 533224536125000, 'EV': 454138663125000, 'EV/EBITDA': 10.039866729960416, 'EV/EBIT': 69.15491439667207, 'P/E': 34.43023781889443}


In [85]:
# Samsung successfully done, now with SK Hynix

result_hynix = get_valuation('000660')
print(result_hynix)

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'
{'ticker': '000660', 'revenue': 32765719000000, 'ebit': -7730313000000, 'ebitda': -7730313000000, 'net_income': -9137547000000, 'market_cap': 103012334647500, 'EV': 94952388647500, 'EV/EBITDA': -12.283123419129343, 'EV/EBIT': -12.283123419129343, 'P/E': -11.27352172826033}


- No D&A again (EBITDA appears same as EBIT)

In [91]:
# SK하이닉스 D&A 파싱 디버깅
df_hynix = dart.finstate('000660', 2023)
rcept_no_hynix = df_hynix['rcept_no'].values[0]
docs_hynix = dart.sub_docs(rcept_no_hynix)

note_url_hynix = docs_hynix[docs_hynix['title'].str.contains('연결재무제표 주석')]['url'].values[0]
response_hynix = requests.get(note_url_hynix)
soup_hynix = BeautifulSoup(response_hynix.content, 'lxml')
tables_hynix = soup_hynix.find_all('table')

for i, table in enumerate(tables_hynix):
    text = table.get_text()
    if '감가상각비' in text and '무형자산상각비' in text:
        print(f"테이블 {i}:")
        df_t = pd.read_html(str(table))[0]
        print(df_t.to_string())
        print()

테이블 547:
         Unnamed: 0       공시금액
0                급여     829260
1              퇴직급여      35537
2             복리후생비     220675
3             지급수수료     769489
4             감가상각비     304389
5           무형자산상각비     282685
6             운반보관비      53680
7             세금과공과      85672
8             광고선전비      83575
9              소모품비     120607
10            판매촉진비     117811
11            품질관리비     146604
12               기타     396174
13               소계    3446158
14  연구개발비 총지출액, 판관비    4101257
15          개발비 자산화  (350,550)
16            경상개발비    3750707
17               합계    7196865

테이블 550:
         Unnamed: 0       공시금액
0                급여    1079387
1              퇴직급여      44341
2             복리후생비     237640
3             지급수수료    1095111
4             감가상각비     264845
5           무형자산상각비     414554
6             운반보관비      58688
7             세금과공과     102035
8             광고선전비     113401
9              소모품비     157009
10            판매촉진비     172715
11            품질관리비 

C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\3396592071.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\3396592071.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\3396592071.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_3556\3396592071.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read 

## The naive approach (writing separate parsing logic per table structure) would require adding conditions indefinitely as new companies are introduced.

## - Instead, a generalized approach
regardless of how a company structures its filing, D&A will always appear as labeled row with a numeric value.
The function searches all tables, extracts any row matching '감가상각비' and '무형자산상각비', and returns the largest combined value.

## Try 2 (Final)

In [ ]:
## 밑 최종 comps def 들 실행 위한 라이브러리 모음집
import OpenDartReader
import FinanceDataReader as fdr
import pandas as pd
import requests
from bs4 import BeautifulSoup
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import os

api_key = 'dfd644cb97b2f1b8d21fa57b07a0d90a0760dd07'
dart = OpenDartReader(api_key)

In [89]:
# D&A 파싱 함수 포함해서 새로운 함수
import requests
from bs4 import BeautifulSoup
import pandas as pd

def find_da(tables):
    best_total = 0
    
    for table in tables:
        text = table.get_text()
        if '감가상각비' not in text or '무형자산상각비' not in text:
            continue
        try:
            df_t = pd.read_html(str(table))[0]
            da = 0
            intangible = 0
            for _, row in df_t.iterrows():
                val = str(row.iloc[-1]).replace(',','').replace('(','').replace(')','').strip()
                if '감가상각비' == str(row.iloc[0]).strip():
                    try: da = abs(int(val)) * 1_000_000
                    except: pass
                if '무형자산상각비' == str(row.iloc[0]).strip():
                    try: intangible = abs(int(val)) * 1_000_000
                    except: pass
            if da + intangible > best_total:
                best_total = da + intangible
        except:
            continue
    
    return best_total


def get_valuation(ticker, year=2023):
    # 1. 재무제표
    df = dart.finstate(ticker, year)
    cfs = df[df['fs_div'] == 'CFS']
    income = cfs[cfs['sj_nm'] == '손익계산서']
    
    df_detail = dart.finstate_all(ticker, year, reprt_code='11011')
    bs_detail = df_detail[df_detail['sj_div'] == 'BS']
    
    # 2. 손익계산서
    revenue = int(income[income['account_nm'] == '매출액']['thstrm_amount'].values[0].replace(',', ''))
    ebit = int(income[income['account_nm'] == '영업이익']['thstrm_amount'].values[0].replace(',', ''))
    net_income = int(income[income['account_nm'].str.contains('당기순이익')]['thstrm_amount'].values[0].replace(',', ''))
    
    # 3. 재무상태표
    def get_amount(df, name):
        row = df[df['account_nm'] == name]
        return int(row['thstrm_amount'].values[0]) if len(row) > 0 else 0
    
    cash = get_amount(bs_detail, '현금및현금성자산')
    short_fin = get_amount(bs_detail, '단기금융상품')
    short_debt = get_amount(bs_detail, '단기차입금')
    current_ltd = get_amount(bs_detail, '유동성장기부채')
    bonds = get_amount(bs_detail, '사채')
    long_debt = get_amount(bs_detail, '장기차입금')
    
    total_debt = short_debt + current_ltd + bonds + long_debt
    net_debt = total_debt - cash - short_fin
    
    # 4. D&A 파싱
    rcept_no = df['rcept_no'].values[0]
    docs = dart.sub_docs(rcept_no)
    note_url = docs[docs['title'].str.contains('연결재무제표 주석')]['url'].values[0]
    response = requests.get(note_url)
    soup = BeautifulSoup(response.content, 'lxml')
    tables = soup.find_all('table')
    da_total = find_da(tables)
    
    # 5. 주가/시총
    stock = fdr.DataReader(ticker, '2023-12-28', '2023-12-28')
    close_price = stock['Close'].values[0]
    shares = dart.report(ticker, '주식총수', year)
    shares_common = int(shares[shares['se'] == '보통주']['istc_totqy'].values[0].replace(',', ''))
    try:
        shares_preferred = int(shares[shares['se'] == '우선주']['istc_totqy'].values[0].replace(',', '').replace('-', '0'))
    except:
        shares_preferred = 0
    market_cap = close_price * (shares_common + shares_preferred)
    
    # 6. EV 및 멀티플
    EV = market_cap + net_debt
    ebitda = ebit + da_total
    
    return {
        'ticker': ticker,
        'revenue': revenue,
        'ebit': ebit,
        'ebitda': ebitda,
        'net_income': net_income,
        'market_cap': market_cap,
        'EV': EV,
        'EV/EBITDA': round(EV / ebitda, 2) if ebitda != 0 else None,
        'EV/EBIT': round(EV / ebit, 2) if ebit != 0 else None,
        'P/E': round(market_cap / net_income, 2) if net_income != 0 else None
    }

In [91]:
result_samsung = get_valuation('005930')
result_hynix = get_valuation('000660')
print(result_samsung)
print(result_hynix)

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'


C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To r

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'


C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To r

{'ticker': '005930', 'revenue': 258935494000000, 'ebit': 6566976000000, 'ebitda': 45674635000000, 'net_income': 15487100000000, 'market_cap': 533224536125000, 'EV': 454138663125000, 'EV/EBITDA': 9.94, 'EV/EBIT': 69.15, 'P/E': 34.43}
{'ticker': '000660', 'revenue': 32765719000000, 'ebit': -7730313000000, 'ebitda': 6421157000000, 'net_income': -9137547000000, 'market_cap': 103012334647500, 'EV': 94952388647500, 'EV/EBITDA': 14.79, 'EV/EBIT': -12.28, 'P/E': -11.27}


Last Check with other company

In [13]:
# 마지막 실험
result_db = get_valuation('000990')
print(result_db)

reprt_code='11011', fs_div='CFS' (사업보고서, 연결제무제표)'


C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_t = pd.read_html(str(table))[0]
C:\Users\cogus\AppData\Local\Temp\ipykernel_21016\1349346429.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To r

{'ticker': '000990', 'revenue': 1154223638940, 'ebit': 265437456373, 'ebitda': 398846456373, 'net_income': 264149250979, 'market_cap': 2608354386200, 'EV': 2075035865817, 'EV/EBITDA': 5.2, 'EV/EBIT': 7.82, 'P/E': 9.87}


### Result: Two functions (`find_da()` for D&A extraction via HTML parsing, and `get_valuation()` which runs the entire pipeline from data collection to valuation metrics.) 
### Input a ticker, get the full output. (`find_da()` is included in `get_valuation()`)

--------------------------------------------------------------------

## Developing another function for Excel Output: Generating IB-Formatted Comps Table
The valuation results are exported into a formatted Excel file 

Library Imports & Directory Setup

In [98]:
import os
os.chdir(r'C:\Users\cogus\OneDrive\바탕 화면\References 26-1\Financial Modeling (Rhee Denis 교수님)')
print(os.getcwd())

C:\Users\cogus\OneDrive\바탕 화면\References 26-1\Financial Modeling (Rhee Denis 교수님)


In [100]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

## Try 1

In [101]:
# 엑셀 결과물 출력 함수
def build_comps_excel(results, company_names, filename='comps.xlsx'):
    # 1. DataFrame 만들기
    df = pd.DataFrame(results)
    df['기업명'] = company_names
    df = df[['기업명', 'ticker', 'revenue', 'market_cap', 'EV', 'ebitda', 'ebit', 'net_income', 'EV/EBITDA', 'EV/EBIT', 'P/E']]
    df.columns = ['기업명', 'Ticker', 'Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income', 'EV/EBITDA', 'EV/EBIT', 'P/E']

    # 2. 평균 행 추가
    avg_row = pd.Series({
        '기업명': 'Average',
        'Ticker': '-',
        'Revenue': df['Revenue'].mean(),
        'Market Cap': df['Market Cap'].mean(),
        'EV': df['EV'].mean(),
        'EBITDA': df['EBITDA'].mean(),
        'EBIT': df['EBIT'].mean(),
        'Net Income': df['Net Income'].mean(),
        'EV/EBITDA': df['EV/EBITDA'].mean(),
        'EV/EBIT': df['EV/EBIT'].mean(),
        'P/E': df['P/E'].mean()
    })
    df = pd.concat([df, avg_row.to_frame().T], ignore_index=True)

    # 3. 엑셀 작성
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Comps'

    blue_fill = PatternFill(fill_type='solid', fgColor='DCE6F1')
    yellow_fill = PatternFill(fill_type='solid', fgColor='FFFF00')
    gray_fill = PatternFill(fill_type='solid', fgColor='F2F2F2')

    headers = list(df.columns)
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Font(bold=True)
        cell.fill = blue_fill
        cell.alignment = Alignment(horizontal='center')

    for row_idx, row in df.iterrows():
        is_avg = row['기업명'] == 'Average'
        for col_idx, value in enumerate(row, 1):
            cell = ws.cell(row=row_idx+2, column=col_idx, value=value)
            cell.alignment = Alignment(horizontal='center')
            if is_avg:
                cell.fill = yellow_fill
                cell.font = Font(bold=True)
            elif row_idx % 2 == 0:
                cell.fill = gray_fill
            if headers[col_idx-1] in ['EV/EBITDA', 'EV/EBIT', 'P/E']:
                cell.number_format = '0.00"x"'

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

    wb.save(filename)
    print(f"✅ {filename} 저장 완료!")
    return df

In [113]:
# 데이터 수집 (앞에서 해둔 거)
results = [result_samsung, result_hynix, result_db]
names = ['Samsung Electronics', 'SK Hynix', 'DB HiTek']

# 엑셀 출력 (이제 위 함수 적용)
df_result = build_comps_excel(results, names)

✅ comps.xlsx 저장 완료!


### Issues with initial version

Two problems identified from the first output:
- Financial figures displayed in scientific notation (e.g. 2.58935E+14) -> will convert to hundreds of millions (억) with comma formatting for readability
- Average multiples were skewed by loss-making companies (e.g. SK Hynix's negative EV/EBIT distorting the mean) -> will revise to exclude negative multiples from the average calculation

## Try 2 (Final)

1. 교수님 피드백 이후 버전 (음수 () 표기)

In [ ]:
def build_comps_excel(results, company_names, filename='comps_final.xlsx'):
    
    # 음수를 () 표기로 바꾸는 함수
    def format_ib(val, is_multiple=False):
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return '-'
        if isinstance(val, str):
            return val
        if is_multiple:
            if val < 0:
                return f'({abs(val):.2f}x)'
            return f'{val:.2f}x'
        else:
            if val < 0:
                return f'({abs(int(val)):,})'
            return f'{int(val):,}'

    # 1. DataFrame 만들기
    df = pd.DataFrame(results)
    df['Company'] = company_names
    df = df[['Company', 'ticker', 'revenue', 'market_cap', 'EV', 'ebitda', 'ebit', 'net_income', 'EV/EBITDA', 'EV/EBIT', 'P/E']]
    df.columns = ['Company', 'Ticker', 'Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income', 'EV/EBITDA', 'EV/EBIT', 'P/E']

    # 억 단위 변환
    for col in ['Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income']:
        df[col] = (df[col] / 1e8).round(0)

    # 2. 평균 행 추가
    avg_row = pd.Series({
        'Company': 'Average',
        'Ticker': '-',
        'Revenue': round(df['Revenue'].mean(), 0),
        'Market Cap': round(df['Market Cap'].mean(), 0),
        'EV': round(df['EV'].mean(), 0),
        'EBITDA': round(df['EBITDA'].mean(), 0),
        'EBIT': round(df['EBIT'].mean(), 0),
        'Net Income': round(df['Net Income'].mean(), 0),
        'EV/EBITDA': round(df[df['EV/EBITDA'] > 0]['EV/EBITDA'].mean(), 2),
        'EV/EBIT': round(df[df['EV/EBIT'] > 0]['EV/EBIT'].mean(), 2),
        'P/E': round(df[df['P/E'] > 0]['P/E'].mean(), 2)
    })
    df = pd.concat([df, avg_row.to_frame().T], ignore_index=True)

    # 3. 엑셀 작성
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Comps'

    blue_fill = PatternFill(fill_type='solid', fgColor='DCE6F1')
    yellow_fill = PatternFill(fill_type='solid', fgColor='FFFF00')
    gray_fill = PatternFill(fill_type='solid', fgColor='F2F2F2')

    # 단위 헤더
    ws.merge_cells('A1:K1')
    unit_cell = ws['A1']
    unit_cell.value = 'Comparable Company Analysis (KRW in hundreds of millions)'
    unit_cell.font = Font(bold=True, italic=True, size=9)
    unit_cell.alignment = Alignment(horizontal='left')

    # 컬럼 헤더
    headers = list(df.columns)
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=2, column=col_idx, value=header)
        cell.font = Font(bold=True)
        cell.fill = blue_fill
        cell.alignment = Alignment(horizontal='center')

    # 데이터
    multiple_cols = ['EV/EBITDA', 'EV/EBIT', 'P/E']
    financial_cols = ['Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income']

    for row_idx, row in df.iterrows():
        is_avg = row['Company'] == 'Average'
        for col_idx, value in enumerate(row, 1):
            col_name = headers[col_idx - 1]
            is_multiple = col_name in multiple_cols
            is_financial = col_name in financial_cols

            if is_multiple or is_financial:
                formatted = format_ib(value, is_multiple=is_multiple)
                cell = ws.cell(row=row_idx + 3, column=col_idx, value=formatted)
            else:
                cell = ws.cell(row=row_idx + 3, column=col_idx, value=value)

            cell.alignment = Alignment(horizontal='center')
            if is_avg:
                cell.fill = yellow_fill
                cell.font = Font(bold=True)
            elif row_idx % 2 == 0:
                cell.fill = gray_fill

    # 열 너비 자동조정
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

    wb.save(filename)
    print(f"✅ {filename} 저장 완료!")
    return df

# 실행
results = [result_samsung, result_hynix, result_db]
names = ['Samsung Electronics', 'SK Hynix', 'DB HiTek']
df_result = build_comps_excel(results, names)

2. 교수님 피드백 이전 (음수 - 표기)

In [25]:
# 결과물 출력 보고 수정 사항 다시 고친 함수

def build_comps_excel(results, company_names, filename='comps_revise.xlsx'):
    df = pd.DataFrame(results)
    df['기업명'] = company_names
    df = df[['기업명', 'ticker', 'revenue', 'market_cap', 'EV', 'ebitda', 'ebit', 'net_income', 'EV/EBITDA', 'EV/EBIT', 'P/E']]
    df.columns = ['기업명', 'Ticker', 'Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income', 'EV/EBITDA', 'EV/EBIT', 'P/E']

    # 억 단위 변환
    for col in ['Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income']:
        df[col] = (df[col] / 1e8).round(0).astype(int)

    # 평균 행 - 양수만 포함
    avg_row = pd.Series({
        '기업명': 'Average',
        'Ticker': '-',
        'Revenue': int(df['Revenue'].mean()),
        'Market Cap': int(df['Market Cap'].mean()),
        'EV': int(df['EV'].mean()),
        'EBITDA': int(df['EBITDA'].mean()),
        'EBIT': int(df['EBIT'].mean()),
        'Net Income': int(df['Net Income'].mean()),
        'EV/EBITDA': round(df[df['EV/EBITDA'] > 0]['EV/EBITDA'].mean(), 2),
        'EV/EBIT': round(df[df['EV/EBIT'] > 0]['EV/EBIT'].mean(), 2),
        'P/E': round(df[df['P/E'] > 0]['P/E'].mean(), 2)
    })
    df = pd.concat([df, avg_row.to_frame().T], ignore_index=True)

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Comps'

    # 단위 헤더 추가
    ws.merge_cells('A1:K1')
    unit_cell = ws['A1']
    unit_cell.value = 'Comparable Company Analysis (KRW in hundreds of millions)'
    unit_cell.font = Font(bold=True, italic=True, size=9)
    unit_cell.alignment = Alignment(horizontal='left')

    blue_fill = PatternFill(fill_type='solid', fgColor='DCE6F1')
    yellow_fill = PatternFill(fill_type='solid', fgColor='FFFF00')
    gray_fill = PatternFill(fill_type='solid', fgColor='F2F2F2')

    headers = list(df.columns)
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=2, column=col_idx, value=header)
        cell.font = Font(bold=True)
        cell.fill = blue_fill
        cell.alignment = Alignment(horizontal='center')

    for row_idx, row in df.iterrows():
        is_avg = row['기업명'] == 'Average'
        for col_idx, value in enumerate(row, 1):
            cell = ws.cell(row=row_idx+3, column=col_idx, value=value)
            cell.alignment = Alignment(horizontal='center')
            if is_avg:
                cell.fill = yellow_fill
                cell.font = Font(bold=True)
            elif row_idx % 2 == 0:
                cell.fill = gray_fill
            if headers[col_idx-1] in ['EV/EBITDA', 'EV/EBIT', 'P/E']:
                cell.number_format = '0.00"x"'
            elif headers[col_idx-1] in ['Revenue', 'Market Cap', 'EV', 'EBITDA', 'EBIT', 'Net Income']:
                cell.number_format = '#,##0'

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

    wb.save(filename)
    print(f"✅ {filename} 저장 완료!")
    return df

In [27]:
# 실행
results = [result_samsung, result_hynix, result_db]
names = ['Samsung Electronics', 'SK Hynix', 'DB HiTek']
df_result = build_comps_excel(results, names)

✅ comps_revise.xlsx 저장 완료!
